# qahirah

qahirah is a pure-Python (ctypes) binding for the Cairo 2D graphics library. It runs on Linux and OpenBSD. Importing it
loads Cairo and FreeType (and Fontconfig, when it is present), so Cairo and FreeType have to be installed.

## Example 1: Import qahirah and read the Cairo version

In [ ]:
# Example 1: Import qahirah and read the Cairo version
import qahirah

version = qahirah.version_tuple()
print(f"Cairo version: {qahirah.version_string()} {version}")
assert len(version) == 3 and all(isinstance(v, int) for v in version), version
assert qahirah.version_string() == ".".join(str(v) for v in version)
assert qahirah.version() == version[0] * 10000 + version[1] * 100 + version[2]

## Example 2: Fill a rectangle on an ImageSurface and read the pixels back

In [ ]:
# Example 2: Fill a rectangle on an ImageSurface and read the pixels back
import ctypes

import qahirah

surface = qahirah.ImageSurface.create(format=qahirah.CAIRO.FORMAT_ARGB32, dimensions=(16, 16))
ctx = qahirah.Context.create(surface)
ctx.set_source_colour(qahirah.Colour.from_rgba((1, 0, 0, 1)))   # opaque red
ctx.rectangle(qahirah.Rect(0, 0, 8, 16))                          # left half of the surface
ctx.fill()
surface.flush()

print(f"width={surface.width} height={surface.height} stride={surface.stride}")
assert (surface.width, surface.height) == (16, 16)
assert surface.stride == qahirah.ImageSurface.format_stride_for_width(qahirah.CAIRO.FORMAT_ARGB32, 16) == 64

# ARGB32 stores one native-endian 32-bit word per pixel; surface.data is the buffer address
pixels = (ctypes.c_uint32 * (surface.stride // 4 * surface.height)).from_address(surface.data)
left, right = pixels[0], pixels[15]
print(f"pixel (0, 0) = {left:#010x}, pixel (15, 0) = {right:#010x}")
assert left == 0xFFFF0000, hex(left)     # filled: alpha 0xff, red 0xff
assert right == 0, hex(right)            # untouched: fully transparent

## Example 3: Write the surface as PNG and load it back

In [ ]:
# Example 3: Write the surface as PNG and load it back
import ctypes

import qahirah

png = surface.to_png_bytes()
print(f"PNG: {len(png)} bytes")
assert png.startswith(b"\x89PNG\r\n\x1a\n")

loaded = qahirah.ImageSurface.create_from_png_bytes(png)
assert tuple(loaded.dimensions) == (16, 16)
assert loaded.format == qahirah.CAIRO.FORMAT_ARGB32
pixels = (ctypes.c_uint32 * (loaded.stride // 4 * loaded.height)).from_address(loaded.data)
assert (pixels[0], pixels[15]) == (0xFFFF0000, 0), (hex(pixels[0]), hex(pixels[15]))
print("PNG round trip kept the pixels")